In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
REPO = "https://github.com/wieland-github/information_extraction_german_medical_data.git"
!rm -rf /content/information_extraction_german_medical_data
!git clone {REPO} /content/information_extraction_german_medical_data
%cd /content/information_extraction_german_medical_data
!ls meaningful_modification/scripts/gptnermed_data_preparation.py   # muss existieren (sonst: Skript-Push fehlt)

In [ ]:
!pip install -q "spacy[cuda12x,transformers]>=3.8.14,<3.9.0" "datasets>=2.16.0,<3.0.0"

## Trainingsdaten bauen: Original + synthetisch

`SYNTHETIC` zeigt auf den zuvor nach Drive kopierten Ordner.
Das Skript hängt die synthetischen Sätze an `train.spacy` an; `validation.spacy`/`test.spacy` bleiben Original.

In [ ]:
%cd /content/information_extraction_german_medical_data
SYNTHETIC = "/content/drive/MyDrive/synthetic_gptnermed.jsonl"
!ls -lh {SYNTHETIC} 

prep = (
    "python meaningful_modification/scripts/gptnermed_data_preparation.py "
    "--outdir meaningful_modification/data "
    f"--synthetic {SYNTHETIC}"
)
print(prep)
!{prep}
!ls -lh meaningful_modification/data


## Training über mehrere Seeds

Für jeden Seed wird ein eigenes Modell trainiert und sofort nach Drive gesichert
(so überstehen fertige Seeds einen Colab-Disconnect).

In [ ]:
import shutil, os

MODEL = "deepset/gbert-large"
TEST  = False                 # True = 50 Steps (Schnelltest), False = volles Training
SEEDS = [42, 52, 54, 62]

base   = MODEL.split("/")[-1] + "-syn" + ("-test" if TEST else "")
extra  = "--training.max_steps 50 --training.eval_frequency 25 --training.patience 0" if TEST else ""
DRIVE  = "/content/drive/MyDrive/gbert_colab_syn"
os.makedirs(DRIVE, exist_ok=True)

%cd /content/information_extraction_german_medical_data

for seed in SEEDS:
    name = f"{base}-seed{seed}"
    OUT  = f"./meaningful_modification/models/{name}"
    cmd = (
        f"python -m spacy train related_work/config_spacy.cfg "
        f"--components.transformer.model.name {MODEL} "
        f"--output {OUT} "
        f"--paths.train ./meaningful_modification/data/train.spacy "
        f"--paths.dev ./meaningful_modification/data/validation.spacy "
        f"--gpu-id 0 "
        f"--system.seed {seed} "
        f"--training.optimizer.learn_rate.initial_rate 5e-5 "
        f"--training.batcher.size 128 {extra}"
    )
    print("\n==================== TRAINING seed", seed, "====================")
    print(cmd)
    !{cmd}

    # bestes Modell dieses Seeds sofort nach Drive sichern
    src = f"/content/information_extraction_german_medical_data/meaningful_modification/models/{name}/model-best"
    dst = f"{DRIVE}/{name}-model-best"
    shutil.make_archive(dst, "zip", src)
    print("Gespeichert in Drive:", dst + ".zip")